In [ ]:
import json
import os
from collections.abc import Iterable
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns


---

1. Анализ каждой модели по всем сидам одного пресета настроек
2. Сравнение пресетов настроек для одной модели
3. Сравнение моделей между собой

In [ ]:
PROJECT_ROOT = Path(os.getcwd()).parent
RESULTS_DIR = PROJECT_ROOT / 'results'
PROMPTS_DIR = PROJECT_ROOT / 'prompts'

In [ ]:
model_results_dir = list(RESULTS_DIR.glob('*'))
model_results_dir

In [ ]:
model_results = {}

for results_dir in model_results_dir:
    
    with open(results_dir / 'metadata.json', "r") as metadata:
        meta = json.load(metadata)
        model_name = meta['model']['name']
    
    # jsonl_run = []
    with open(results_dir / 'runs.jsonl', "r") as run:
        jsonl_df = pd.read_json(run, lines=True)
        # for json_str in run:
        #     json_line = json.loads(json_str)
        #     jsonl_run.append(json_line)
        
    model_results.update({
        model_name: {
            "metadata": meta,
            "run_info": jsonl_df
        }
    })


---

Model -> Prompt -> settings

In [ ]:
model_results.keys()

In [ ]:
df_dict = {}

for model_name in model_results:
    df_dict[model_name] = deepcopy(model_results[model_name]['run_info'])
    # df_dict[model_name] = df_dict[model_name].drop(columns=['model'])
    df_dict[model_name].head()

In [ ]:
df_models = pd.concat([*df_dict.values()])
df_models.head()

In [ ]:
def group_by_prompt_and_mode(
    df_model: pd.DataFrame, 
    by: Iterable[str] = ("model", "prompt", "mode"),
):
    prompt_analysis = {}

    if type(by) != 'list':
        by = list(by)

    for (model, prompt, mode), df_slice in df_model.groupby(by=by):
        metrics = pd.DataFrame.from_records(df_slice["metrics"].values)
        timings = pd.DataFrame.from_records(df_slice["timings"].values)
        usage = pd.DataFrame.from_records(df_slice["usage"].values)
        response = pd.DataFrame.from_records(df_slice["response"].values)

        new_col = pd.DataFrame.from_records(usage["prompt_tokens_details"])
        usage = usage.drop(columns=["prompt_tokens_details"])
        usage = pd.concat([usage, new_col], axis=1)

        for df_tmp in [metrics, timings, usage, response]:
            df_tmp.insert(0, "model", model)

        if response["response_id"].nunique() == response.shape[0]:
            response = response.drop(columns=["response_id"])

        prompt_analysis \
            .setdefault(model, {}) \
            .setdefault(prompt, {})[mode] = {
                "metrics": metrics,
                "timings": timings,
                "usage": usage,
                "response": response,
            }

    return prompt_analysis

In [ ]:
model_prompt_settings_df_dict = {}

model_prompt_settings_df_dict = group_by_prompt_and_mode(df_models)

In [ ]:
model_prompt_settings_df_dict['gemma4']['p1_generation']['conservative']['metrics']

---

Prompt -> settings -> model

In [ ]:
def combine_models(data: dict) -> dict:
    combined = {}

    for prompts in data.values():
        for prompt, modes in prompts.items():
            for mode, measurements in modes.items():
                for measurement, df in measurements.items():
                    combined \
                        .setdefault(prompt, {}) \
                        .setdefault(mode, {}) \
                        .setdefault(measurement, []) \
                        .append(df)

    for prompt, modes in combined.items():
        for mode, measurements in modes.items():
            for measurement, dfs in measurements.items():
                combined[prompt][mode][measurement] = pd.concat(
                    dfs,
                    ignore_index=True,
                )

    return combined

In [ ]:
prompt_settings_df_dict = combine_models(model_prompt_settings_df_dict)

In [ ]:
prompt_settings_df_dict['p1_generation']['conservative']['usage']

---

In [ ]:
prompt_settings_stat_df_dict = {}

for prompt, modes in prompt_settings_df_dict.items():
    for mode, measurements in modes.items():
        for measurement, df in measurements.items():
            if measurement != 'response':
                stats_df = df.groupby(by=['model']).agg(['mean', 'std'])
                prompt_settings_stat_df_dict \
                    .setdefault(prompt, {}) \
                    .setdefault(mode, {})[measurement] = stats_df
            else:
                if prompt == 'p2_classification':
                    # Не очевидный костыль, но я таким образом добавляю столбец меток классов сразу и в dataframe старого словаря
                    df['classes'] = \
                        pd.Series([
                            [r.split(' ')[1] for r in rr] 
                            for rr in [rrr.lower().strip().split('\n') for rrr in df['text'].values]
                        ])
                prompt_settings_stat_df_dict \
                    .setdefault(prompt, {}) \
                    .setdefault(mode, {})[measurement] = df

In [ ]:
prompt_settings_stat_df_dict['p2_classification']['conservative']['metrics']

---

## Classification

In [ ]:
with open(PROMPTS_DIR / 'p2_answers.txt', 'r', encoding='utf-8') as f:
    clf_answers = [s.lower().strip().split(' ')[1] for s in f]

clf_answers

In [ ]:
for mode in prompt_settings_stat_df_dict['p2_classification']:
    df_response = prompt_settings_stat_df_dict['p2_classification'][mode]['response']
    
    acc_new_col, n_correct_new_col = [], []
    acc_metric_results = {}

    for model_name, df_slice in df_response.groupby(by=['model']):
        model_name = model_name[0]
        
        pred = df_slice['classes']
        pred_matrix = np.array(clf_answers)[None, :] == np.array(pred.values.tolist())
        
        model_acc = np.mean(pred_matrix, axis=1)
        model_mean_acc = np.mean(model_acc)
        model_std_acc = np.std(model_acc)
        
        n_correct = np.sum(pred_matrix, axis=1)
        n_correct_mean = np.mean(n_correct)
        n_correct_std = np.std(n_correct)

        acc_metric_results[model_name] = {
            "accuracy": {
                "mean": model_mean_acc,
                "std": model_std_acc,
            },

            "n_correct": {
                "mean": n_correct_mean,
                "std": n_correct_std,
            },
        }

        acc_new_col.append(model_acc)
        n_correct_new_col.append(n_correct)

    acc_new_col = np.r_[*acc_new_col]
    n_correct_new_col = np.r_[*n_correct_new_col]

    df_response['accuracy'] = acc_new_col
    df_response['n_correct'] = n_correct_new_col

    prompt_settings_df_dict['p2_classification'][mode]['response']['accuracy'] = acc_new_col
    prompt_settings_df_dict['p2_classification'][mode]['response']['n_correct'] = n_correct_new_col

    # Add new 2-level column into metrics (pandas approach)
    stats_df = pd.DataFrame({
        (metric, stat): {
            model: values[metric][stat]
            for model, values in acc_metric_results.items()
        }
        for metric in ["accuracy", "n_correct"]
        for stat in ["mean", "std"]
    })
    stats_df.index.name = "model"
    prompt_settings_stat_df_dict['p2_classification'][mode]['metrics'] \
        = pd.concat([prompt_settings_stat_df_dict['p2_classification'][mode]['metrics'], stats_df], axis=1)

In [ ]:
prompt_settings_stat_df_dict['p2_classification']['top_k_focused']['response']

In [ ]:
display(prompt_settings_stat_df_dict['p2_classification']['baseline']['response'])

----

## Анализ

In [ ]:
modes_map = {
    "baseline": {
        "temperature": 1.0,
        "top_p": 0.95,
        "top_k": 64
    },

    "conservative": {
        "temperature": 0.2,
        "top_p": 0.8,
    },

    "top_k_focused": {
        "temperature": 0.5,
        "top_k": 20,
    },

    "creative": {
        "temperature": 1.1,
        "top_p": 0.98,
    },
}

In [ ]:
print(prompt_settings_df_dict['p1_generation']['baseline']['response'])